In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = sns.load_dataset("penguins").dropna()

# Pairplot
pairplot = sns.pairplot(df,hue="species",diag_kind="kde",palette="Set2")
plt.suptitle("Pairplot of Penguin Measurements by Species", y=1.02)
pairplot.savefig("penguins_pairplot.png", dpi=300, bbox_inches="tight")
plt.close()

from google.colab import files
files.download("penguins_pairplot.png")

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import scipy.stats as stats
from google.colab import files

# Load the penguin dataset
penguins = sns.load_dataset("penguins")
penguins = penguins.dropna(subset=['flipper_length_mm', 'body_mass_g'])

# Feature and target
X = penguins[['flipper_length_mm']]
y = penguins['body_mass_g']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_pred = lr_model.predict(X_test)

# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Regression Plot with RMSE and R²
plt.figure(figsize=(8,6))
plt.scatter(X_test, y_test, color='blue', label='Actual Body Mass')
plt.plot(X_test, y_pred, color='red', label='Fitted Line')
plt.title('Linear Regression: Flipper Length vs. Body Mass')
plt.xlabel('Flipper Length (mm)')
plt.ylabel('Body Mass (g)')
plt.legend()

# Annotate metrics on the plot
plt.text(
    0.05, 0.95,
    f"RMSE = {rmse:.2f}\nR² = {r2:.3f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    bbox=dict(facecolor='white', alpha=0.7, edgecolor='black')
)

plt.savefig("penguins_regression.png", dpi=300, bbox_inches="tight")
plt.close()
files.download("penguins_regression.png")

# Residual Plot
residuals = y_test - y_pred
plt.figure(figsize=(8,6))
plt.scatter(X_test, residuals, color='green')
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residual Plot')
plt.xlabel('Flipper Length (mm)')
plt.ylabel('Residuals')

plt.savefig("penguins_residuals.png", dpi=300, bbox_inches="tight")
plt.close()
files.download("penguins_residuals.png")

# Q-Q Plot
plt.figure(figsize=(6,6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot of Residuals')

plt.savefig("penguins_qqplot.png", dpi=300, bbox_inches="tight")
plt.close()
files.download("penguins_qqplot.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score
)

# Load and prepare data
penguins = sns.load_dataset("penguins")
penguins = penguins.dropna(subset=['bill_length_mm', 'bill_depth_mm',
                                   'flipper_length_mm', 'body_mass_g', 'sex'])

# Encode target: male = 1, female = 0
penguins['is_male'] = (penguins['sex'] == 'Male').astype(int)

# Features and target
feature_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X = penguins[feature_cols].copy()
y = penguins['is_male']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit logistic regression
log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(X_train, y_train)

# Predictions and probabilities
y_pred = log_reg_model.predict(X_test)
y_pred_prob = log_reg_model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Helper: 2D decision boundary (white background, no shading)
def plot_decision_boundary(ax, model, df, target, feature_pair, fixed_values=None):
    f1, f2 = feature_pair
    x_min, x_max = df[f1].min(), df[f1].max()
    y_min, y_max = df[f2].min(), df[f2].max()
    x_range = np.linspace(x_min, x_max, 200)
    y_range = np.linspace(y_min, y_max, 200)
    xx, yy = np.meshgrid(x_range, y_range)

    if fixed_values is None:
        fixed_values = df.median(numeric_only=True).to_dict()

    grid = pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()})
    for col in df.columns:
        if col not in [f1, f2]:
            grid[col] = fixed_values[col]

    zz = model.predict_proba(grid[df.columns])[:, 1].reshape(xx.shape)

    # Only plot the decision boundary (p=0.5), no shading
    ax.contour(xx, yy, zz, levels=[0.5], colors='k', linewidths=2)

    # Scatter actual points (explicit colors, no cmap background)
    colors = ['red' if val == 1 else 'blue' for val in target]
    ax.scatter(df[f1], df[f2], c=colors, edgecolor='k', s=35, alpha=0.8)

    ax.set_xlabel(f1.replace('_mm', ' (mm)').replace('_g', ' (g)'))
    ax.set_ylabel(f2.replace('_mm', ' (mm)').replace('_g', ' (g)'))
    ax.set_title(f'Decision boundary: {f1} vs {f2}')

# Plot: Confusion, ROC, Decision Boundary
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Confusion matrix
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=['Female (0)', 'Male (1)'],
            yticklabels=['Female (0)', 'Male (1)'])
axes[0].set_title('Confusion matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Middle: ROC curve with metrics box
axes[1].plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='red', linestyle='--', label='Baseline')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC curve: Sex classification')
axes[1].legend(loc='lower right')
metrics_text = (
    f"Accuracy:  {accuracy:.3f}\n"
    f"Precision: {precision:.3f}\n"
    f"Recall:    {recall:.3f}\n"
    f"F1-score:  {f1:.3f}\n"
    f"AUC:       {roc_auc:.3f}"
)
axes[1].text(0.58, 0.20, metrics_text, transform=axes[1].transAxes,
             fontsize=10, bbox=dict(facecolor='white', alpha=0.85, edgecolor='black'))

# Right: Decision boundary for a chosen feature pair
feature_pair = ('bill_depth_mm', 'body_mass_g')  # you can change this to any two features
plot_decision_boundary(
    axes[2], log_reg_model, X_test, y_test, feature_pair,
    fixed_values=X_test.median(numeric_only=True).to_dict()
)

plt.tight_layout()
plt.savefig("penguins_classification.png", dpi=300, bbox_inches="tight")
plt.close()

from google.colab import files
files.download("penguins_classification.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score
)

from google.colab import files

# Load and prepare data
penguins = sns.load_dataset("penguins")
penguins = penguins.dropna(subset=['bill_length_mm', 'bill_depth_mm',
                                   'flipper_length_mm', 'body_mass_g', 'sex'])

# Encode target: male = 1, female = 0
penguins['is_male'] = (penguins['sex'] == 'Male').astype(int)

# Features and target
feature_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X = penguins[feature_cols].copy()
y = penguins['is_male']

# Train/test split (fixed random_state for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define models with fixed random_state
models = {
    "Non-Regularized": LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000, random_state=42),
    "L2 (Ridge)": LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000, random_state=42),
    "L1 (Lasso)": LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42),
    "Elastic Net": LogisticRegression(penalty='elasticnet', solver='saga',
                                      l1_ratio=0.5, max_iter=5000, random_state=42)
}

results = {}
predictions = {}
probas = {}

# Fit and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = y_pred
    probas[name] = y_pred_prob

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_prob)
    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc}

# Convert results to DataFrame
results_df = pd.DataFrame(results).T
print(results_df)

# Performance Metrics Plot
ax = results_df.plot(kind="bar", figsize=(10,6))
plt.title("Performance Comparison of Logistic Regression Regularization")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="best")
plt.tight_layout()
plt.savefig("performance_metrics.png", dpi=300, bbox_inches="tight")
plt.close()
files.download("performance_metrics.png")

# Combined ROC Curve for all models
plt.figure(figsize=(8,6))
for name, y_pred_prob in probas.items():
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    auc = roc_auc_score(y_test, y_pred_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.2f})")
plt.plot([0,1],[0,1],'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Logistic Regression Models")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves_all_models.png", dpi=300, bbox_inches="tight")
plt.close()
files.download("roc_curves_all_models.png")

# 2D decision boundary
def plot_decision_boundary(ax, model, df, target, feature_pair, fixed_values=None):
    f1, f2 = feature_pair
    x_min, x_max = df[f1].min(), df[f1].max()
    y_min, y_max = df[f2].min(), df[f2].max()
    x_range = np.linspace(x_min, x_max, 200)
    y_range = np.linspace(y_min, y_max, 200)
    xx, yy = np.meshgrid(x_range, y_range)

    if fixed_values is None:
        fixed_values = df.median(numeric_only=True).to_dict()

    grid = pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()})
    for col in df.columns:
        if col not in [f1, f2]:
            grid[col] = fixed_values[col]

    zz = model.predict_proba(grid[df.columns])[:, 1].reshape(xx.shape)

    ax.contour(xx, yy, zz, levels=[0.5], colors='k', linewidths=2)
    colors = ['red' if val == 1 else 'blue' for val in target]
    ax.scatter(df[f1], df[f2], c=colors, edgecolor='k', s=35, alpha=0.8)

    ax.set_xlabel(f1.replace('_mm', ' (mm)').replace('_g', ' (g)'))
    ax.set_ylabel(f2.replace('_mm', ' (mm)').replace('_g', ' (g)'))

# Side-by-side Confusion Matrix + Decision Boundary for each model
feature_pair = ('bill_depth_mm', 'body_mass_g')

for name, model in models.items():
    y_pred = predictions[name]

    fig, axes = plt.subplots(1, 2, figsize=(12,5))

    # Confusion Matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=['Female (0)', 'Male (1)'],
                yticklabels=['Female (0)', 'Male (1)'])
    axes[0].set_title(f"{name} Confusion Matrix")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    # Decision Boundary
    plot_decision_boundary(
        axes[1], model, X_test, y_test, feature_pair,
        fixed_values=X_test.median(numeric_only=True).to_dict()
    )
    axes[1].set_title(f"{name} Decision Boundary")

    plt.tight_layout()
    fname = f"{name.replace(' ','_').lower()}_confusion_decision.png"
    plt.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close()
    files.download(fname)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


                 Accuracy  Precision    Recall        F1       AUC
Non-Regularized  0.865672   0.837838  0.911765  0.873239  0.936720
L2 (Ridge)       0.776119   0.787879  0.764706  0.776119  0.882353
L1 (Lasso)       0.820896   0.843750  0.794118  0.818182  0.902852
Elastic Net      0.686567   0.685714  0.705882  0.695652  0.800357


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>